# Combine every patient FHIR JSON file

This notebook recursively reads every `.json` and `.ndjson` file in `fir-2` (or `fhir-2`), unwraps FHIR Bundles, combines resources across patients, and creates cohort-level and one-row-per-patient datasets.

It expects FHIR R4-style resources such as `Patient`, `MedicationRequest`, `Condition`, `Encounter`, and `Observation`. A `MedicationRequest` is an order/prescription; it does not prove that a medicine was dispensed or taken.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import json

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# The override is useful for testing; normally leave it unset.
folder_override = globals().get("FHIR_FOLDER_OVERRIDE")
candidates = [Path("fir-2"), Path("fhir-2"), Path("data/fir-2"), Path("data/fhir-2")]
FHIR_FOLDER = Path(folder_override) if folder_override else next((p for p in candidates if p.exists()), Path("fir-2"))
OUTPUT_FOLDER = Path("output/combined_fhir")
FAIL_ON_INVALID_JSON = False
IS_SYNTHETIC = True  # Change to False for real patient data.
WRITE_OUTPUTS = True

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 100)
print(f"Input folder: {FHIR_FOLDER.resolve()}")

/Users/daimon/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Input folder: /Users/daimon/Documents/climate_health_hack/fhir-2


## 1. Read every file and unwrap every Bundle

The load report lets you confirm that all files were processed. Invalid files are recorded without stopping the cohort unless `FAIL_ON_INVALID_JSON` is enabled.

In [2]:
def resources_from_document(document):
    if not isinstance(document, dict):
        return []
    if document.get("resourceType") == "Bundle":
        return [
            entry["resource"]
            for entry in document.get("entry", [])
            if isinstance(entry.get("resource"), dict) and entry["resource"].get("resourceType")
        ]
    return [document] if document.get("resourceType") else []


def documents_from_file(file_path):
    text = file_path.read_text(encoding="utf-8-sig")
    try:
        yield json.loads(text)
        return
    except json.JSONDecodeError:
        pass

    # If the full file was not JSON, try one FHIR resource per line (NDJSON).
    for line_number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        try:
            yield json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON on line {line_number}") from exc


def load_fhir_folder(folder, fail_on_invalid=False):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(
            f"FHIR folder not found: {folder.resolve()}\n"
            "Place fir-2 beside this notebook, or change FHIR_FOLDER in the configuration cell."
        )

    files = sorted(p for p in folder.rglob("*") if p.suffix.lower() in {".json", ".ndjson"})
    if not files:
        raise FileNotFoundError(f"No JSON or NDJSON files found under {folder.resolve()}")

    all_resources = []
    file_rows = []
    for file_number, file_path in enumerate(files, start=1):
        relative_name = str(file_path.relative_to(folder))
        file_resources = []
        error = None
        try:
            for document in documents_from_file(file_path):
                file_resources.extend(resources_from_document(document))
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
            if fail_on_invalid:
                raise

        patient_ids = sorted({r.get("id") for r in file_resources if r.get("resourceType") == "Patient" and r.get("id")})
        for resource in file_resources:
            all_resources.append({"resource": resource, "source_file": relative_name})
        file_rows.append({
            "source_file": relative_name,
            "status": "error" if error else "loaded",
            "resource_count": len(file_resources),
            "patient_count": len(patient_ids),
            "patient_ids": " | ".join(patient_ids),
            "error": error,
        })
        if file_number % 100 == 0:
            print(f"Processed {file_number:,}/{len(files):,} files...")

    return all_resources, pd.DataFrame(file_rows)

In [3]:
resource_records, file_load_report = load_fhir_folder(FHIR_FOLDER, FAIL_ON_INVALID_JSON)
resource_counts = Counter(item["resource"].get("resourceType", "Unknown") for item in resource_records)
resource_count_table = pd.DataFrame(resource_counts.most_common(), columns=["resource_type", "count"])

print(f"Files discovered: {len(file_load_report):,}")
print(f"Files loaded: {(file_load_report['status'] == 'loaded').sum():,}")
print(f"Resources combined: {len(resource_records):,}")
display(file_load_report.head(20))
display(resource_count_table)

Processed 100/557 files...
Processed 200/557 files...
Processed 300/557 files...
Processed 400/557 files...
Processed 500/557 files...
Files discovered: 557
Files loaded: 557
Resources combined: 409,973


,source_file,status,resource_count,patient_count,patient_ids,error
0,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json,loaded,744,1,b0a06ead-cc42-aa48-dad6-841d4aa679fa,None
1,Abe604_Schmidt332_ccfc4db2-2026-7adb-3db0-33f3828140bb.json,loaded,557,1,ccfc4db2-2026-7adb-3db0-33f3828140bb,None
2,Adelaida985_DuBuque211_31a2e8ec-69fc-8a71-3ab6-36cbdd508713.json,loaded,692,1,31a2e8ec-69fc-8a71-3ab6-36cbdd508713,None
3,Adriana394_Hintz995_71a8b156-760b-df6b-859e-eefc7932a526.json,loaded,330,1,71a8b156-760b-df6b-859e-eefc7932a526,None
4,Agueda283_Gerhold939_76b289fd-e825-734c-8446-316f59643593.json,loaded,933,1,76b289fd-e825-734c-8446-316f59643593,None
5,Ahmed109_O'Reilly797_92fb7efc-5cfd-f8d3-927b-42f8ee099531.json,loaded,442,1,92fb7efc-5cfd-f8d3-927b-42f8ee099531,None
6,Aimee901_Hudson301_81aa7647-779f-fd6b-94cf-782e606efeb2.json,loaded,308,1,81aa7647-779f-fd6b-94cf-782e606efeb2,None
7,Akiko835_Pfannerstill264_346a1435-2455-914f-c287-7b88052d05db.json,loaded,988,1,346a1435-2455-914f-c287-7b88052d05db,None
8,Alaine226_Willms744_1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4.json,loaded,712,1,1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,None
9,Alberta625_Waters156_97899f1d-9c3b-2b90-17b8-400c11ab8f0f.json,loaded,627,1,97899f1d-9c3b-2b90-17b8-400c11ab8f0f,None


,resource_type,count
0,Observation,131703
1,Claim,52068
2,DiagnosticReport,45020
3,Procedure,38528
4,Encounter,27812
5,DocumentReference,27812
6,ExplanationOfBenefit,27812
7,MedicationRequest,24256
8,Condition,17253
9,Immunization,8100


## 2. Convert nested FHIR into combined tables

In [4]:
def reference_id(value):
    reference = value.get("reference") if isinstance(value, dict) else value
    if not reference:
        return None
    return str(reference).rstrip("/").rsplit("/", 1)[-1].rsplit(":", 1)[-1]


def first_coding(concept):
    concept = concept or {}
    coding = next((x for x in concept.get("coding", []) if isinstance(x, dict)), {})
    return coding.get("code"), concept.get("text") or coding.get("display"), coding.get("system")


def first_name(resource):
    names = resource.get("name", [])
    item = next((x for x in names if x.get("use") == "official"), names[0] if names else {})
    given = " ".join(item.get("given", []))
    family = item.get("family", "")
    return f"{given} {family}".strip() or None


def address_coordinates(address):
    latitude = longitude = None
    for extension in address.get("extension", []):
        url = extension.get("url", "").lower()
        if url.endswith("geolocation"):
            for child in extension.get("extension", []):
                child_url = child.get("url", "").lower()
                if child_url.endswith("latitude"):
                    latitude = child.get("valueDecimal")
                elif child_url.endswith("longitude"):
                    longitude = child.get("valueDecimal")
    return latitude, longitude


def observation_value(resource):
    quantity = resource.get("valueQuantity")
    if quantity:
        return quantity.get("value"), quantity.get("unit") or quantity.get("code")
    for key in ("valueString", "valueBoolean", "valueInteger", "valueDateTime"):
        if key in resource:
            return resource[key], None
    code, display_text, _ = first_coding(resource.get("valueCodeableConcept"))
    return display_text or code, None

In [5]:
rows = defaultdict(list)
medication_definitions = {}
for item in resource_records:
    resource = item["resource"]
    if resource.get("resourceType") == "Medication":
        medication_definitions[resource.get("id")] = first_coding(resource.get("code"))

for item in resource_records:
    resource = item["resource"]
    source = item["source_file"]
    resource_type = resource.get("resourceType")
    last_updated = resource.get("meta", {}).get("lastUpdated")

    if resource_type == "Patient":
        addresses = resource.get("address", [])
        address = next((x for x in addresses if x.get("use") == "home"), addresses[0] if addresses else {})
        latitude, longitude = address_coordinates(address)
        rows["patients"].append({
            "patient_id": resource.get("id"), "name": first_name(resource), "gender": resource.get("gender"),
            "birth_date": resource.get("birthDate"), "deceased_date": resource.get("deceasedDateTime"),
            "city": address.get("city"), "state": address.get("state"), "county": address.get("district"),
            "postal_code": address.get("postalCode"), "latitude": latitude, "longitude": longitude,
            "last_updated": last_updated, "source_file": source,
        })

    elif resource_type == "MedicationRequest":
        code, display_text, system = first_coding(resource.get("medicationCodeableConcept"))
        if not code and resource.get("medicationReference"):
            medication_id = reference_id(resource["medicationReference"])
            code, resolved_display, system = medication_definitions.get(medication_id, (medication_id, None, "FHIR reference"))
            display_text = resource["medicationReference"].get("display") or resolved_display
        dosage = resource.get("dosageInstruction", [])
        rows["medications"].append({
            "medication_request_id": resource.get("id"), "patient_id": reference_id(resource.get("subject")),
            "encounter_id": reference_id(resource.get("encounter")), "status": resource.get("status"),
            "intent": resource.get("intent"), "authored_on": resource.get("authoredOn"),
            "medication_code": code, "medication_display": display_text, "medication_system": system,
            "dosage_text": dosage[0].get("text") if dosage else None,
            "last_updated": last_updated, "source_file": source,
        })

    elif resource_type == "Condition":
        code, display_text, system = first_coding(resource.get("code"))
        clinical_status, _, _ = first_coding(resource.get("clinicalStatus"))
        rows["conditions"].append({
            "condition_id": resource.get("id"), "patient_id": reference_id(resource.get("subject")),
            "clinical_status": clinical_status, "condition_code": code, "condition_display": display_text,
            "condition_system": system,
            "onset_date": resource.get("onsetDateTime") or resource.get("onsetPeriod", {}).get("start"),
            "abatement_date": resource.get("abatementDateTime") or resource.get("abatementPeriod", {}).get("end"),
            "last_updated": last_updated, "source_file": source,
        })

    elif resource_type == "Encounter":
        encounter_class = resource.get("class", {})
        period = resource.get("period", {})
        rows["encounters"].append({
            "encounter_id": resource.get("id"), "patient_id": reference_id(resource.get("subject")),
            "status": resource.get("status"), "class_code": encounter_class.get("code"),
            "class_display": encounter_class.get("display"), "start_date": period.get("start"),
            "end_date": period.get("end"), "last_updated": last_updated, "source_file": source,
        })

    elif resource_type == "Observation":
        code, display_text, system = first_coding(resource.get("code"))
        value, unit = observation_value(resource)
        rows["observations"].append({
            "observation_id": resource.get("id"), "patient_id": reference_id(resource.get("subject")),
            "encounter_id": reference_id(resource.get("encounter")), "status": resource.get("status"),
            "effective_date": resource.get("effectiveDateTime") or resource.get("effectivePeriod", {}).get("start"),
            "observation_code": code, "observation_display": display_text, "observation_system": system,
            "value": value, "unit": unit, "last_updated": last_updated, "source_file": source,
        })

patients = pd.DataFrame(rows["patients"])
medications = pd.DataFrame(rows["medications"])
conditions = pd.DataFrame(rows["conditions"])
encounters = pd.DataFrame(rows["encounters"])
observations = pd.DataFrame(rows["observations"])

print({name: len(frame) for name, frame in {"patients": patients, "medications": medications, "conditions": conditions, "encounters": encounters, "observations": observations}.items()})

{'patients': 555, 'medications': 24256, 'conditions': 17253, 'encounters': 27812, 'observations': 131703}


## 3. Clean, deduplicate, and validate patient links

In [6]:
def clean_table(frame, id_column, date_columns):
    frame = frame.copy()
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        frame[column] = frame[column].map(lambda x: x.strip() if isinstance(x, str) else x).replace("", pd.NA)
    for column in date_columns:
        if column in frame:
            frame[column] = pd.to_datetime(frame[column], errors="coerce", utc=True)
    if "last_updated" in frame:
        frame["last_updated"] = pd.to_datetime(frame["last_updated"], errors="coerce", utc=True)
        frame = frame.sort_values("last_updated", na_position="first")
    return frame.drop_duplicates(id_column, keep="last").reset_index(drop=True)


patients = clean_table(patients, "patient_id", ["birth_date", "deceased_date"])
medications = clean_table(medications, "medication_request_id", ["authored_on"])
conditions = clean_table(conditions, "condition_id", ["onset_date", "abatement_date"])
encounters = clean_table(encounters, "encounter_id", ["start_date", "end_date"])
observations = clean_table(observations, "observation_id", ["effective_date"])

patients["postal_code"] = patients["postal_code"].astype("string")
patients["zip3"] = patients["postal_code"].str.extract(r"(\d{3})", expand=False)
patients["latitude"] = pd.to_numeric(patients["latitude"], errors="coerce")
patients["longitude"] = pd.to_numeric(patients["longitude"], errors="coerce")

known_patients = set(patients["patient_id"].dropna())
quality_rows = []
for table_name, frame in [("MedicationRequest", medications), ("Condition", conditions), ("Encounter", encounters), ("Observation", observations)]:
    missing_ref = int(frame["patient_id"].isna().sum())
    unmatched_ref = int((frame["patient_id"].notna() & ~frame["patient_id"].isin(known_patients)).sum())
    quality_rows.append({"resource_type": table_name, "rows": len(frame), "missing_patient_reference": missing_ref, "unmatched_patient_reference": unmatched_ref})
quality_report = pd.DataFrame(quality_rows)
display(quality_report)

,resource_type,rows,missing_patient_reference,unmatched_patient_reference
0,MedicationRequest,24256,0,0
1,Condition,17253,0,0
2,Encounter,27812,0,0
3,Observation,131703,0,0


## 4. Build one combined row for each patient

`patient_overview` combines demographics, location, event counts, condition/medication lists, and each patient's observed time range.

In [7]:
def patient_counts(frame, value_column, count_name, unique_name=None):
    grouped = frame.groupby("patient_id").agg(**{count_name: (value_column, "size")})
    if unique_name:
        grouped[unique_name] = frame.groupby("patient_id")[value_column].nunique(dropna=True)
    return grouped.reset_index()


patient_overview = patients.copy()
for summary in [
    patient_counts(conditions, "condition_id", "condition_count"),
    patient_counts(medications, "medication_code", "medication_request_count", "unique_medication_count"),
    patient_counts(encounters, "encounter_id", "encounter_count"),
    patient_counts(observations, "observation_id", "observation_count"),
]:
    patient_overview = patient_overview.merge(summary, on="patient_id", how="left", validate="one_to_one")

condition_lists = (conditions.dropna(subset=["condition_display"]).groupby("patient_id")["condition_display"]
                   .agg(lambda s: " | ".join(sorted(set(s)))).rename("conditions").reset_index())
medication_lists = (medications.dropna(subset=["medication_display"]).groupby("patient_id")["medication_display"]
                    .agg(lambda s: " | ".join(sorted(set(s)))).rename("medications").reset_index())
patient_overview = patient_overview.merge(condition_lists, on="patient_id", how="left", validate="one_to_one")
patient_overview = patient_overview.merge(medication_lists, on="patient_id", how="left", validate="one_to_one")

activity_parts = []
for frame, date_column in [(conditions, "onset_date"), (medications, "authored_on"), (encounters, "start_date"), (observations, "effective_date")]:
    activity_parts.append(frame[["patient_id", date_column]].rename(columns={date_column: "activity_date"}))
activity = pd.concat(activity_parts, ignore_index=True).dropna(subset=["patient_id", "activity_date"])
activity_range = activity.groupby("patient_id")["activity_date"].agg(first_activity="min", last_activity="max").reset_index()
patient_overview = patient_overview.merge(activity_range, on="patient_id", how="left", validate="one_to_one")

for column in ["condition_count", "medication_request_count", "unique_medication_count", "encounter_count", "observation_count"]:
    patient_overview[column] = patient_overview[column].fillna(0).astype(int)
patient_overview["has_diabetes"] = patient_overview["conditions"].str.contains("diabet", case=False, na=False)
patient_overview["has_neuropathy"] = patient_overview["conditions"].str.contains("neuropath", case=False, na=False)
patient_overview["has_pain_related_condition"] = patient_overview["conditions"].str.contains(
    r"pain|migraine|gout|arthritis|fibromyalgia|neuropath", case=False, na=False, regex=True
)

print(f"Combined patient rows: {len(patient_overview):,}")
display(patient_overview.head(20))

Combined patient rows: 555


,patient_id,name,gender,birth_date,deceased_date,city,state,county,postal_code,latitude,longitude,last_updated,source_file,zip3,condition_count,medication_request_count,unique_medication_count,encounter_count,observation_count,conditions,medications,first_activity,last_activity,has_diabetes,has_neuropathy,has_pain_related_condition
0,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Abdul218 Harris789,male,1952-12-05 00:00:00+00:00,2002-02-01 08:14:47+00:00,Boston,MA,None,02134,42.377994,-71.075704,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json,021,38,53,7,32,289,Acute bronchitis (disorder) | Anemia (disorder) | Concussion with no loss of consciousness | Cor...,Acetaminophen 325 MG Oral Tablet | Amlodipine 5 MG Oral Tablet | Atorvastatin 80 MG Oral Tablet ...,1954-04-03 17:14:47+00:00,2002-02-08 08:14:47+00:00,True,False,False
1,ccfc4db2-2026-7adb-3db0-33f3828140bb,Abe604 Schmidt332,male,1976-06-06 00:00:00+00:00,NaT,Peabody,MA,None,01940,42.534889,-70.959732,NaT,Abe604_Schmidt332_ccfc4db2-2026-7adb-3db0-33f3828140bb.json,019,38,49,4,38,138,Acute bronchitis (disorder) | Acute viral pharyngitis (disorder) | Body mass index 30+ - obesity...,72 HR Fentanyl 0.025 MG/HR Transdermal System | Acetaminophen 300 MG / Hydrocodone Bitartrate 5 ...,1994-07-31 05:20:43+00:00,2021-09-26 12:20:43+00:00,False,False,True
2,31a2e8ec-69fc-8a71-3ab6-36cbdd508713,Adelaida985 DuBuque211,female,1917-05-15 00:00:00+00:00,2017-02-18 08:58:49+00:00,Quincy,MA,None,02169,42.282390,-71.024155,NaT,Adelaida985_DuBuque211_31a2e8ec-69fc-8a71-3ab6-36cbdd508713.json,021,78,1,1,64,175,Acute bacterial sinusitis (disorder) | Acute viral pharyngitis (disorder) | Alzheimer's disease ...,Donepezil hydrochloride 23 MG Oral Tablet,1935-07-09 11:58:49+00:00,2017-02-21 11:58:49+00:00,False,False,False
3,71a8b156-760b-df6b-859e-eefc7932a526,Adriana394 Hintz995,female,2014-11-15 00:00:00+00:00,NaT,Boston,MA,None,02120,42.398050,-70.971345,NaT,Adriana394_Hintz995_71a8b156-760b-df6b-859e-eefc7932a526.json,021,3,4,3,19,175,Laceration of forearm | Otitis media | Streptococcal sore throat (disorder),Cefuroxime 250 MG Oral Tablet | Ibuprofen 100 MG Oral Tablet | Penicillin V Potassium 250 MG Ora...,2014-11-15 09:49:18+00:00,2021-11-06 09:49:18+00:00,False,False,False
4,76b289fd-e825-734c-8446-316f59643593,Agueda283 Gerhold939,female,1985-03-03 00:00:00+00:00,NaT,Boston,MA,None,02115,42.377613,-71.030448,NaT,Agueda283_Gerhold939_76b289fd-e825-734c-8446-316f59643593.json,021,30,12,10,24,564,Acute deep venous thrombosis (disorder) | Acute viral pharyngitis (disorder) | Anemia (disorder)...,0.4 ML Enoxaparin sodium 100 MG/ML Prefilled Syringe | 1 ML Enoxaparin sodium 150 MG/ML Prefille...,1989-11-16 05:40:38+00:00,2021-10-03 05:40:38+00:00,True,False,False
5,92fb7efc-5cfd-f8d3-927b-42f8ee099531,Ahmed109 O'Reilly797,male,2013-06-12 00:00:00+00:00,NaT,Worcester,MA,None,01604,42.254165,-71.700925,NaT,Ahmed109_O'Reilly797_92fb7efc-5cfd-f8d3-927b-42f8ee099531.json,016,10,7,5,27,216,COVID-19 | Concussion with no loss of consciousness | Fever (finding) | Fracture subluxation of ...,Acetaminophen 160 MG Chewable Tablet | Amoxicillin 500 MG Oral Tablet | Ibuprofen 100 MG Oral Ta...,2013-06-12 04:56:06+00:00,2021-06-09 04:56:06+00:00,False,False,False
6,81aa7647-779f-fd6b-94cf-782e606efeb2,Aimee901 Hudson301,female,2014-12-29 00:00:00+00:00,NaT,Weston,MA,None,<NA>,42.384528,-71.331503,NaT,Aimee901_Hudson301_81aa7647-779f-fd6b-94cf-782e606efeb2.json,<NA>,1,1,1,17,174,Acute bronchitis (disorder),Acetaminophen 325 MG Oral Tablet,2014-12-30 03:31:56+00:00,2020-12-15 03:31:56+00:00,False,False,False
7,346a1435-2455-914f-c287-7b88052d05db,Akiko835 Pfannerstill264,female,1981-11-16 00:00:00+00:00,NaT,Framingham,MA,None,01702,42.318855,-71.400519,NaT,Akiko835_Pfannerstill264_346a1435-2455-914f-c287-7b88052d05db.json,017,33,57,8,69,241,Acute viral pharyngitis (disorder) | Anemia (disorder) | Fracture of forearm | Full-time employm...,1 ML medroxyprogesterone acetate 150 MG/ML Injection | Astemizo

## 5. Overall cohort data

In [8]:
overall_summary = pd.DataFrame([
    {"metric": "JSON/NDJSON files", "value": len(file_load_report)},
    {"metric": "Files with load errors", "value": int((file_load_report["status"] == "error").sum())},
    {"metric": "Unique patients", "value": patients["patient_id"].nunique()},
    {"metric": "Medication requests", "value": len(medications)},
    {"metric": "Conditions", "value": len(conditions)},
    {"metric": "Encounters", "value": len(encounters)},
    {"metric": "Observations", "value": len(observations)},
    {"metric": "Patients with diabetes-related text", "value": int(patient_overview["has_diabetes"].sum())},
    {"metric": "Patients with neuropathy-related text", "value": int(patient_overview["has_neuropathy"].sum())},
    {"metric": "Patients with pain-related condition text", "value": int(patient_overview["has_pain_related_condition"].sum())},
])

top_medications = (medications.assign(medication_display=medications["medication_display"].fillna("Unknown"))
                   .groupby(["medication_code", "medication_display"], dropna=False)
                   .agg(requests=("medication_request_id", "count"), patients=("patient_id", "nunique"))
                   .sort_values(["patients", "requests"], ascending=False).reset_index())
top_conditions = (conditions.assign(condition_display=conditions["condition_display"].fillna("Unknown"))
                  .groupby(["condition_code", "condition_display"], dropna=False)
                  .agg(records=("condition_id", "count"), patients=("patient_id", "nunique"))
                  .sort_values(["patients", "records"], ascending=False).reset_index())
location_summary = (patient_overview.assign(state=patient_overview["state"].fillna("Unknown"))
                    .groupby("state").agg(patients=("patient_id", "nunique"),
                                                medication_requests=("medication_request_count", "sum"),
                                                conditions=("condition_count", "sum"),
                                                encounters=("encounter_count", "sum"))
                    .sort_values("patients", ascending=False).reset_index())

display(overall_summary)
display(top_conditions.head(20))
display(top_medications.head(20))
display(location_summary.head(20))
display(patient_overview[["condition_count", "medication_request_count", "encounter_count", "observation_count"]].describe())

,metric,value
0,JSON/NDJSON files,557
1,Files with load errors,0
2,Unique patients,555
3,Medication requests,24256
4,Conditions,17253
5,Encounters,27812
6,Observations,131703
7,Patients with diabetes-related text,165
8,Patients with neuropathy-related text,9
9,Patients with pain-related condition text,175


,condition_code,condition_display,records,patients
0,73595000,Stress (finding),2246,416
1,160903007,Full-time employment (finding),6379,389
2,444814009,Viral sinusitis (disorder),585,351
3,160904001,Part-time employment (finding),976,318
4,422650009,Social isolation (finding),515,300
5,423315002,Limited social contact (finding),535,296
6,224299000,Received higher education (finding),257,257
7,195662009,Acute viral pharyngitis (disorder),347,252
8,741062008,Not in labor force (finding),421,226
9,10509002,Acute bronchitis (disorder),266,218


,medication_code,medication_display,requests,patients
0,313782,Acetaminophen 325 MG Oral Tablet,285,227
1,849574,Naproxen sodium 220 MG Oral Tablet,126,111
2,562251,Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet,125,109
3,314076,lisinopril 10 MG Oral Tablet,3777,99
4,310798,Hydrochlorothiazide 25 MG Oral Tablet,3281,88
5,308136,amLODIPine 2.5 MG Oral Tablet,2816,76
6,310965,Ibuprofen 200 MG Oral Tablet,79,71
7,1043400,Acetaminophen 21.7 MG/ML / Dextromethorphan Hydrobromide 1 MG/ML / doxylamine succinate 0.417 MG...,64,61
8,1870230,NDA020800 0.3 ML Epinephrine 1 MG/ML Auto-Injector,59,59
9,198405,Ibuprofen 100 MG Oral Tablet,67,56


,state,patients,medication_requests,conditions,encounters
0,MA,555,24256,17253,27812


,condition_count,medication_request_count,encounter_count,observation_count
count,555.000000,555.000000,555.000000,555.000000
mean,31.086486,43.704505,50.111712,237.302703
std,35.213937,141.282276,81.876338,422.248689
min,0.000000,0.000000,2.000000,10.000000
25%,11.500000,3.000000,20.000000,120.000000
50%,25.000000,9.000000,38.000000,153.000000
75%,42.500000,41.000000,60.000000,240.500000
max,460.000000,2117.000000,1563.000000,7869.000000


## 6. Inspect one patient across all combined files

Set `PATIENT_TO_VIEW` to a specific FHIR Patient ID, or leave it as `None` to show the first patient.

In [9]:
PATIENT_TO_VIEW = None
if PATIENT_TO_VIEW is None and not patients.empty:
    PATIENT_TO_VIEW = patients.iloc[0]["patient_id"]

def show_patient(patient_id):
    print(f"Patient: {patient_id}")
    display(patient_overview[patient_overview["patient_id"] == patient_id])
    display(conditions[conditions["patient_id"] == patient_id].sort_values("onset_date"))
    display(medications[medications["patient_id"] == patient_id].sort_values("authored_on"))
    display(encounters[encounters["patient_id"] == patient_id].sort_values("start_date"))
    display(observations[observations["patient_id"] == patient_id].sort_values("effective_date"))

if PATIENT_TO_VIEW:
    show_patient(PATIENT_TO_VIEW)

Patient: b0a06ead-cc42-aa48-dad6-841d4aa679fa


,patient_id,name,gender,birth_date,deceased_date,city,state,county,postal_code,latitude,longitude,last_updated,source_file,zip3,condition_count,medication_request_count,unique_medication_count,encounter_count,observation_count,conditions,medications,first_activity,last_activity,has_diabetes,has_neuropathy,has_pain_related_condition
0,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Abdul218 Harris789,male,1952-12-05 00:00:00+00:00,2002-02-01 08:14:47+00:00,Boston,MA,None,02134,42.377994,-71.075704,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json,021,38,53,7,32,289,Acute bronchitis (disorder) | Anemia (disorder) | Concussion with no loss of consciousness | Cor...,Acetaminophen 325 MG Oral Tablet | Amlodipine 5 MG Oral Tablet | Atorvastatin 80 MG Oral Tablet ...,1954-04-03 17:14:47+00:00,2002-02-08 08:14:47+00:00,True,False,False


,condition_id,patient_id,clinical_status,condition_code,condition_display,condition_system,onset_date,abatement_date,last_updated,source_file
0,8695c578-79e7-b7fa-0d3b-be91b95d4a56,b0a06ead-cc42-aa48-dad6-841d4aa679fa,active,224299000,Received higher education (finding),http://snomed.info/sct,1971-01-29 09:04:48+00:00,NaT,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
1,2f9b9ad8-6dd5-b94b-e7e7-d329d204cedb,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,160903007,Full-time employment (finding),http://snomed.info/sct,1971-01-29 09:04:48+00:00,1972-02-04 08:46:16+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
2,c21907ce-d314-d9ac-87b5-e9dff5371c13,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,160903007,Full-time employment (finding),http://snomed.info/sct,1978-02-10 08:45:33+00:00,1981-02-13 09:03:53+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
3,896c3120-7689-0630-2698-04d5a3f5f704,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,160903007,Full-time employment (finding),http://snomed.info/sct,1981-02-13 09:03:53+00:00,1982-04-30 08:59:31+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
4,204b175e-3004-a395-0d11-cc2123bebd43,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,160903007,Full-time employment (finding),http://snomed.info/sct,1984-02-17 08:47:12+00:00,1987-02-20 09:04:53+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
5,4ac39567-65bb-0214-39b9-be6d2fd8793a,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,423315002,Limited social contact (finding),http://snomed.info/sct,1984-02-17 08:47:12+00:00,1993-04-30 08:59:37+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
6,4fa1910c-cb03-a42f-ceae-aedd59fdb024,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,160903007,Full-time employment (finding),http://snomed.info/sct,1987-02-20 09:04:53+00:00,1990-02-23 09:11:45+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
7,a9713149-09b4-1712-50af-61bff2353415,b0a06ead-cc42-aa48-dad6-841d4aa679fa,active,15777000,Prediabetes,http://snomed.info/sct,1990-02-23 08:14:47+00:00,NaT,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
9,c70fe0e1-e212-f3fb-0f12-a75e845dd816,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,73595000,Stress (finding),http://snomed.info/sct,1990-02-23 09:11:45+00:00,1992-04-24 09:01:19+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
8,335254e6-cb3b-2f1e-dee3-c2ec39d0ebf7,b0a06ead-cc42-aa48-dad6-841d4aa679fa,resolved,160903007,Full-time employment (finding),http://snomed.info/sct,1990-02-23 09:11:45+00:00,1992-04-10 09:04:36+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json


,medication_request_id,patient_id,encounter_id,status,intent,authored_on,medication_code,medication_display,medication_system,dosage_text,last_updated,source_file
0,c0e50da8-7aed-f2db-f893-efe740504150,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,stopped,order,1992-04-10 08:14:47+00:00,312961,Simvastatin 20 MG Oral Tablet,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
1,28fe47d9-82cf-c35d-468d-39ecd61c3b9c,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,stopped,order,1992-04-10 08:14:47+00:00,705129,Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
2,4d93f56b-3db5-e0a6-a4c7-f65cf2b8538d,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,stopped,order,1992-04-10 08:14:47+00:00,705129,Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
3,9f5c47fb-b639-e236-ea1a-b6bc2fcab325,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,stopped,order,1992-04-10 08:14:47+00:00,312961,Simvastatin 20 MG Oral Tablet,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
4,e7a52e7d-d8ec-5824-b4a6-bb936b2f299a,b0a06ead-cc42-aa48-dad6-841d4aa679fa,f2f337b4-33bb-1fe1-4f96-8102f9a8b75a,stopped,order,1992-04-24 08:14:47+00:00,705129,Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
5,44157e8e-12e7-dd8a-8025-bf679c750596,b0a06ead-cc42-aa48-dad6-841d4aa679fa,f2f337b4-33bb-1fe1-4f96-8102f9a8b75a,stopped,order,1992-04-24 08:14:47+00:00,312961,Simvastatin 20 MG Oral Tablet,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
6,c1008266-fabb-9917-6092-675baad7e38e,b0a06ead-cc42-aa48-dad6-841d4aa679fa,edf391a1-4331-38fe-ff87-7f6c41ceb8ab,stopped,order,1993-04-30 08:14:47+00:00,705129,Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
7,ea4269fb-72db-13cc-d011-9b22ae6ad6e3,b0a06ead-cc42-aa48-dad6-841d4aa679fa,edf391a1-4331-38fe-ff87-7f6c41ceb8ab,stopped,order,1993-04-30 08:14:47+00:00,312961,Simvastatin 20 MG Oral Tablet,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
14,93036dd5-ad00-f144-76ee-211ceda47686,b0a06ead-cc42-aa48-dad6-841d4aa679fa,f672abad-3594-cef8-9103-99582de0b2bc,stopped,order,1994-05-06 08:14:47+00:00,197361,Amlodipine 5 MG Oral Tablet,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
12,ed3c13dc-3e11-d5dc-6834-ddccc35f4bab,b0a06ead-cc42-aa48-dad6-841d4aa679fa,f672abad-3594-cef8-9103-99582de0b2bc,stopped,order,1994-05-06 08:14:47+00:00,705129,Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray,http://www.nlm.nih.gov/research/umls/rxnorm,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json


,encounter_id,patient_id,status,class_code,class_display,start_date,end_date,last_updated,source_file
0,a889805b-20ca-0ba6-5b5c-e617ce99b0cc,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1954-04-03 17:14:47+00:00,1954-04-03 17:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
1,b1ca5d4d-e3b5-e65b-e900-3844ae30f421,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1954-04-20 13:14:47+00:00,1954-04-20 13:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
2,853a4e65-da0d-d531-eb6e-458a70bcf552,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1959-06-03 19:14:47+00:00,1959-06-03 19:33:08+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
3,b04db379-2092-c969-8968-39ab3557b605,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1971-01-29 08:14:47+00:00,1971-01-29 08:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
4,dffdcaf9-384a-3472-867c-2f852dc61851,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1977-10-25 08:14:47+00:00,1977-10-25 08:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
5,ed03c4f1-9865-7638-37d1-d7c1fe282363,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1978-02-10 08:14:47+00:00,1978-02-10 08:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
6,e756ebcd-40d3-59f6-257c-4b413d4e94e8,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1981-02-13 08:14:47+00:00,1981-02-13 08:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
7,e47502eb-e574-2058-f1f9-4f19844e4da8,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1982-04-22 22:14:47+00:00,1982-04-22 22:37:23+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
8,e44915f6-9666-521e-7c74-6f5a5035db88,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1984-02-17 08:14:47+00:00,1984-02-17 08:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
9,1fff2e1c-2dc4-c32d-4894-c1a62c7c80be,b0a06ead-cc42-aa48-dad6-841d4aa679fa,finished,AMB,None,1987-02-20 08:14:47+00:00,1987-02-20 08:29:47+00:00,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json


,observation_id,patient_id,encounter_id,status,effective_date,observation_code,observation_display,observation_system,value,unit,last_updated,source_file
0,33648be3-5286-a324-312c-c9cf3cfedac8,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,final,1992-04-10 08:14:47+00:00,8302-2,Body Height,http://loinc.org,186,cm,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
27,16adcc64-aae0-e7e2-e4f8-cbb57d165612,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,final,1992-04-10 08:14:47+00:00,4548-4,Hemoglobin A1c/Hemoglobin.total in Blood,http://loinc.org,5.89,%,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
26,7a8eedbc-22a4-6c6c-9839-cbfe8e8da382,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,final,1992-04-10 08:14:47+00:00,72166-2,Tobacco smoking status NHIS,http://loinc.org,Former smoker,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
25,2a98a1c7-7053-9d37-3040-13e541715cfe,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,final,1992-04-10 08:14:47+00:00,32623-1,Platelet mean volume [Entitic volume] in Blood by Automated count,http://loinc.org,9.8167,fL,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
24,7dd75818-0e80-5a41-8dcf-a48826c2c662,b0a06ead-cc42-aa48-dad6-841d4aa679fa,d331b9e7-cc03-6a74-ecac-64fedddb57a9,final,1992-04-10 08:14:47+00:00,32207-3,Platelet distribution width [Entitic volume] in Blood by Automated count,http://loinc.org,299.67,fL,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
...,...,...,...,...,...,...,...,...,...,...,...,...
269,59c574a6-8180-71e8-7a95-b8f7250a675f,b0a06ead-cc42-aa48-dad6-841d4aa679fa,14982877-7fb5-2997-06ac-3cc952d52f9f,final,2001-12-21 08:14:47+00:00,8302-2,Body Height,http://loinc.org,186,cm,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
277,c12c187d-2d5f-b486-707b-ae16dedade38,b0a06ead-cc42-aa48-dad6-841d4aa679fa,14982877-7fb5-2997-06ac-3cc952d52f9f,final,2001-12-21 08:14:47+00:00,6299-2,Urea Nitrogen,http://loinc.org,14.97,mg/dL,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
286,8912421f-8a07-9e20-60e8-0a4823006eb9,b0a06ead-cc42-aa48-dad6-841d4aa679fa,14982877-7fb5-2997-06ac-3cc952d52f9f,final,2001-12-21 08:58:50+00:00,93025-5,"Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]",http://loinc.org,None,None,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json
287,0533ada3-c39c-806d-c0c6-daba3e9d31ba,b0a06ead-cc42-aa48-dad6-841d4aa679fa,14982877-7fb5-2997-06ac-3cc952d52f9f,final,2001-12-21 09:30:33+00:00,75626-2,Total score [AUDIT-C],http://loinc.org,0,{score},NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json


In [20]:
patient_overview.head()

,patient_id,name,gender,birth_date,deceased_date,city,state,county,postal_code,latitude,longitude,last_updated,source_file,zip3,condition_count,medication_request_count,unique_medication_count,encounter_count,observation_count,conditions,medications,first_activity,last_activity,has_diabetes,has_neuropathy,has_pain_related_condition
0,b0a06ead-cc42-aa48-dad6-841d4aa679fa,Abdul218 Harris789,male,1952-12-05 00:00:00+00:00,2002-02-01 08:14:47+00:00,Boston,MA,None,02134,42.377994,-71.075704,NaT,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json,021,38,53,7,32,289,Acute bronchitis (disorder) | Anemia (disorder) | Concussion with no loss of consciousness | Cor...,Acetaminophen 325 MG Oral Tablet | Amlodipine 5 MG Oral Tablet | Atorvastatin 80 MG Oral Tablet ...,1954-04-03 17:14:47+00:00,2002-02-08 08:14:47+00:00,True,False,False
1,ccfc4db2-2026-7adb-3db0-33f3828140bb,Abe604 Schmidt332,male,1976-06-06 00:00:00+00:00,NaT,Peabody,MA,None,01940,42.534889,-70.959732,NaT,Abe604_Schmidt332_ccfc4db2-2026-7adb-3db0-33f3828140bb.json,019,38,49,4,38,138,Acute bronchitis (disorder) | Acute viral pharyngitis (disorder) | Body mass index 30+ - obesity...,72 HR Fentanyl 0.025 MG/HR Transdermal System | Acetaminophen 300 MG / Hydrocodone Bitartrate 5 ...,1994-07-31 05:20:43+00:00,2021-09-26 12:20:43+00:00,False,False,True
2,31a2e8ec-69fc-8a71-3ab6-36cbdd508713,Adelaida985 DuBuque211,female,1917-05-15 00:00:00+00:00,2017-02-18 08:58:49+00:00,Quincy,MA,None,02169,42.282390,-71.024155,NaT,Adelaida985_DuBuque211_31a2e8ec-69fc-8a71-3ab6-36cbdd508713.json,021,78,1,1,64,175,Acute bacterial sinusitis (disorder) | Acute viral pharyngitis (disorder) | Alzheimer's disease ...,Donepezil hydrochloride 23 MG Oral Tablet,1935-07-09 11:58:49+00:00,2017-02-21 11:58:49+00:00,False,False,False
3,71a8b156-760b-df6b-859e-eefc7932a526,Adriana394 Hintz995,female,2014-11-15 00:00:00+00:00,NaT,Boston,MA,None,02120,42.398050,-70.971345,NaT,Adriana394_Hintz995_71a8b156-760b-df6b-859e-eefc7932a526.json,021,3,4,3,19,175,Laceration of forearm | Otitis media | Streptococcal sore throat (disorder),Cefuroxime 250 MG Oral Tablet | Ibuprofen 100 MG Oral Tablet | Penicillin V Potassium 250 MG Ora...,2014-11-15 09:49:18+00:00,2021-11-06 09:49:18+00:00,False,False,False
4,76b289fd-e825-734c-8446-316f59643593,Agueda283 Gerhold939,female,1985-03-03 00:00:00+00:00,NaT,Boston,MA,None,02115,42.377613,-71.030448,NaT,Agueda283_Gerhold939_76b289fd-e825-734c-8446-316f59643593.json,021,30,12,10,24,564,Acute deep venous thrombosis (disorder) | Acute viral pharyngitis (disorder) | Anemia (disorder)...,0.4 ML Enoxaparin sodium 100 MG/ML Prefilled Syringe | 1 ML Enoxaparin sodium 150 MG/ML Prefille...,1989-11-16 05:40:38+00:00,2021-10-03 05:40:38+00:00,True,False,False


## 7. Export the combined cohort

Synthetic patient-level tables are written to `output/combined_fhir`. If this is real patient data, set `IS_SYNTHETIC = False`; the notebook will export aggregate summaries only.

In [11]:
# if WRITE_OUTPUTS:
#     OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
#     file_load_report.to_csv(OUTPUT_FOLDER / "file_load_report.csv", index=False)
#     overall_summary.to_csv(OUTPUT_FOLDER / "overall_summary.csv", index=False)
#     top_conditions.to_csv(OUTPUT_FOLDER / "top_conditions.csv", index=False)
#     top_medications.to_csv(OUTPUT_FOLDER / "top_medications.csv", index=False)
#     location_summary.to_csv(OUTPUT_FOLDER / "location_summary.csv", index=False)
#     quality_report.to_csv(OUTPUT_FOLDER / "quality_report.csv", index=False)

#     if IS_SYNTHETIC:
#         patients.to_csv(OUTPUT_FOLDER / "patients_combined.csv", index=False)
#         medications.to_csv(OUTPUT_FOLDER / "medication_requests_combined.csv", index=False)
#         conditions.to_csv(OUTPUT_FOLDER / "conditions_combined.csv", index=False)
#         encounters.to_csv(OUTPUT_FOLDER / "encounters_combined.csv", index=False)
#         observations.to_csv(OUTPUT_FOLDER / "observations_combined.csv", index=False)
#         patient_overview.to_csv(OUTPUT_FOLDER / "patient_overview_combined.csv", index=False)
#     else:
#         print("Real-data mode: patient-level exports were skipped. Review disclosure controls before exporting them.")

#     print(f"Combined outputs written to: {OUTPUT_FOLDER.resolve()}")
# else:
#     print("No files written because WRITE_OUTPUTS is False.")